In [1]:
! pip install torchreid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=dd42ced074d2719354f4243c4885b3fce0fcd5ef1f4fecca48759600c8cb8672
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid


In [2]:
! unzip data/archive.zip -d data/market1501

Выходные данные были обрезаны до нескольких последних строк (5000).
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c2s2_164677_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c2s2_164677_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c3s3_006062_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c3s3_006062_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c4s5_038360_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c4s5_038360_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c5s3_006593_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c5s3_006593_00_junk.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c6s3_029892_00_good.mat  
  inflating: data/market1501/Market-1501-v15.09.15/gt_query/1180_c6s3_029892_00_junk.mat  
  inflating: data/mark

In [3]:
import torchreid

/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [4]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
datamanager = torchreid.data.ImageDataManager(
    root="data",
    sources="market1501",
    targets="market1501",
    height=256,
    width=128,
    batch_size_train=32,
    batch_size_test=100,
    transforms=["random_flip", "random_crop", "random_erase"]
)

Building train transforms ...
+ resize to 256x128
+ random flip
+ random crop (enlarge to 288x144 and crop 256x128)
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
+ random erase
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------
=> Loading test (target) dataset


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------


  **************** Summary ****************
  source            : ['market1501']
  # source datasets : 1
  # source ids      : 751
  # source images   : 12936
  # source cameras  : 6
  target            : ['market1501']
  *****************************************




In [6]:
model = torchreid.models.build_model(
    name="resnet50",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True
)

model = model.cuda()

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /root/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s]


In [7]:
optimizer = torchreid.optim.build_optimizer(
    model,
    optim="adam",
    lr=0.0003
)

scheduler = torchreid.optim.build_lr_scheduler(
    optimizer,
    lr_scheduler="single_step",
    stepsize=20
)

In [8]:
engine = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    label_smooth=True
)

In [9]:
engine.run(
    save_dir="log/market1501_baseline",
    max_epoch=60,
    eval_freq=10,
    print_freq=10,
    test_only=False
)

# Save clean inference weights (state_dict only)
from pathlib import Path
out_dir = Path('models')
out_dir.mkdir(parents=True, exist_ok=True)

model_to_save = model.module if hasattr(model, 'module') else model
clean_weights_path = out_dir / 'resnet50_market1501_clean.pth'
torch.save(model_to_save.state_dict(), clean_weights_path)
print(f'Clean weights saved: {clean_weights_path}')


=> Start training


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


epoch: [1/60][10/404]	time 0.185 (0.454)	data 0.001 (0.088)	eta 3:03:15	loss 7.2160 (6.8474)	acc 0.0000 (0.6250)	lr 0.000300
epoch: [1/60][20/404]	time 0.185 (0.319)	data 0.001 (0.044)	eta 2:08:47	loss 6.9303 (6.9319)	acc 3.1250 (0.6250)	lr 0.000300
epoch: [1/60][30/404]	time 0.185 (0.274)	data 0.001 (0.030)	eta 1:50:39	loss 6.7450 (6.8684)	acc 0.0000 (0.6250)	lr 0.000300
epoch: [1/60][40/404]	time 0.184 (0.252)	data 0.001 (0.022)	eta 1:41:34	loss 6.6796 (6.8275)	acc 0.0000 (0.4688)	lr 0.000300
epoch: [1/60][50/404]	time 0.188 (0.239)	data 0.001 (0.018)	eta 1:36:14	loss 6.6647 (6.7868)	acc 0.0000 (0.4375)	lr 0.000300
epoch: [1/60][60/404]	time 0.227 (0.233)	data 0.001 (0.015)	eta 1:33:55	loss 6.5944 (6.7581)	acc 0.0000 (0.4688)	lr 0.000300
epoch: [1/60][70/404]	time 0.189 (0.229)	data 0.002 (0.013)	eta 1:32:13	loss 6.6725 (6.7408)	acc 0.0000 (0.4911)	lr 0.000300
epoch: [1/60][80/404]	time 0.187 (0.224)	data 0.000 (0.012)	eta 1:30:23	loss 6.5355 (6.7242)	acc 0.0000 (0.5078)	lr 0.000300


In [13]:
! zip -r model.zip models

  adding: models/ (stored 0%)
  adding: models/resnet50_market1501_clean.pth (deflated 18%)


In [14]:
from google.colab import files
files.download('model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>